In [ ]:
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError("Cannot find repo root containing data/ and notebooks/")

REPO_ROOT = find_repo_root(Path.cwd())
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = REPO_ROOT / "data" / "processed"


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [18]:
customer = pd.read_csv(RAW_DATA_DIR / "olist_customers_dataset.csv")
seller = pd.read_csv(RAW_DATA_DIR / "olist_sellers_dataset.csv")

In [19]:
seller[seller.duplicated(subset= None, keep = 'first')]

,seller_id,seller_zip_code_prefix,seller_city,seller_state


In [20]:
customer_count = customer.customer_state.value_counts()
# use value_counts() to generate the counts and put into DataFrame
customer_count_df = pd.DataFrame(customer_count).reset_index()

# set up the name of column
customer_count_df.columns = ['customer_state', 'cust_state_fre']
customer_count_df

,customer_state,cust_state_fre
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


In [21]:
seller_count = seller.seller_state.value_counts()
# use value_counts() to generate the counts and put into DataFrame
seller_count_df = pd.DataFrame(seller_count).reset_index()

# set up the name of column
seller_count_df.columns = ['seller_state', 'seller_state_fre']
seller_count_df

,seller_state,seller_state_fre
0,SP,1849
1,PR,349
2,MG,244
3,SC,190
4,RJ,171
5,RS,129
6,GO,40
7,DF,30
8,ES,23
9,BA,19


In [22]:
seller_customer_count = pd.merge(customer_count_df,seller_count_df, how='outer', left_on='customer_state', right_on='seller_state')
seller_customer_count.drop('seller_state', axis=1, inplace=True)
seller_customer_count.rename(columns={'customer_state': 'state_brief'}, inplace=True)
seller_customer_count

,state_brief,cust_state_fre,seller_state_fre
0,AC,81,1.0
1,AL,413,NaN
2,AM,148,1.0
3,AP,68,NaN
4,BA,3380,19.0
5,CE,1336,13.0
6,DF,2140,30.0
7,ES,2033,23.0
8,GO,2020,40.0
9,MA,747,1.0


In [23]:
seller_customer_count.fillna(0.1, inplace=True)

In [24]:
seller_customer_count

,state_brief,cust_state_fre,seller_state_fre
0,AC,81,1.0
1,AL,413,0.1
2,AM,148,1.0
3,AP,68,0.1
4,BA,3380,19.0
5,CE,1336,13.0
6,DF,2140,30.0
7,ES,2033,23.0
8,GO,2020,40.0
9,MA,747,1.0


In [25]:
# input the full name of state, as powerbi can't recognize some brief name of state
state_mapping = {
    'AC': 'Acre',
    'AL': 'Alagoas',
    'AP': 'Amapá',
    'AM': 'Amazonas',
    'BA': 'Bahia',
    'CE': 'Ceará',
    'DF': 'Distrito Federal',
    'ES': 'Espírito Santo',
    'GO': 'Goiás',
    'MA': 'Maranhão',
    'MT': 'Mato Grosso',
    'MS': 'Mato Grosso do Sul',
    'MG': 'Minas Gerais',
    'PA': 'Pará',
    'PB': 'Paraíba',
    'PR': 'Paraná',
    'PE': 'Pernambuco',
    'PI': 'Piauí',
    'RJ': 'Rio de Janeiro',
    'RN': 'Rio Grande do Norte',
    'RS': 'Rio Grande do Sul',
    'RO': 'Rondônia',
    'RR': 'Roraima',
    'SC': 'Santa Catarina',
    'SP': 'São Paulo',
    'SE': 'Sergipe',
    'TO': 'Tocantins'
}

seller_customer_count['state_full_name'] = seller_customer_count['state_brief'].map(state_mapping)

display(seller_customer_count)

,state_brief,cust_state_fre,seller_state_fre,state_full_name
0,AC,81,1.0,Acre
1,AL,413,0.1,Alagoas
2,AM,148,1.0,Amazonas
3,AP,68,0.1,Amapá
4,BA,3380,19.0,Bahia
5,CE,1336,13.0,Ceará
6,DF,2140,30.0,Distrito Federal
7,ES,2033,23.0,Espírito Santo
8,GO,2020,40.0,Goiás
9,MA,747,1.0,Maranhão


In [26]:
display(seller_customer_count)

,state_brief,cust_state_fre,seller_state_fre,state_full_name
0,AC,81,1.0,Acre
1,AL,413,0.1,Alagoas
2,AM,148,1.0,Amazonas
3,AP,68,0.1,Amapá
4,BA,3380,19.0,Bahia
5,CE,1336,13.0,Ceará
6,DF,2140,30.0,Distrito Federal
7,ES,2033,23.0,Espírito Santo
8,GO,2020,40.0,Goiás
9,MA,747,1.0,Maranhão


In [27]:
seller_customer_count['customer_seller_ratio'] = seller_customer_count.cust_state_fre / seller_customer_count.seller_state_fre
seller_customer_count['seller_customer_ratio'] = seller_customer_count.seller_state_fre / seller_customer_count.cust_state_fre

In [28]:
seller_customer_count['customer_seller_ratio'] = seller_customer_count['customer_seller_ratio'].round(3)
seller_customer_count['seller_customer_ratio'] = seller_customer_count['seller_customer_ratio'].round(3)
seller_customer_count

,state_brief,cust_state_fre,seller_state_fre,state_full_name,customer_seller_ratio,seller_customer_ratio
0,AC,81,1.0,Acre,81.000,0.012
1,AL,413,0.1,Alagoas,4130.000,0.000
2,AM,148,1.0,Amazonas,148.000,0.007
3,AP,68,0.1,Amapá,680.000,0.001
4,BA,3380,19.0,Bahia,177.895,0.006
5,CE,1336,13.0,Ceará,102.769,0.010
6,DF,2140,30.0,Distrito Federal,71.333,0.014
7,ES,2033,23.0,Espírito Santo,88.391,0.011
8,GO,2020,40.0,Goiás,50.500,0.020
9,MA,747,1.0,Maranhão,747.000,0.001


In [29]:
# export file
seller_customer_count.to_csv(PROCESSED_DATA_DIR / "customer_seller_ratio.csv", index=False)

In [30]:
seller_customer_count.sort_values('customer_seller_ratio')

,state_brief,cust_state_fre,seller_state_fre,state_full_name,customer_seller_ratio,seller_customer_ratio
17,PR,5045,349.0,Paraná,14.456,0.069
23,SC,3637,190.0,Santa Catarina,19.142,0.052
25,SP,41746,1849.0,São Paulo,22.578,0.044
22,RS,5466,129.0,Rio Grande do Sul,42.372,0.024
10,MG,11635,244.0,Minas Gerais,47.684,0.021
8,GO,2020,40.0,Goiás,50.500,0.020
6,DF,2140,30.0,Distrito Federal,71.333,0.014
18,RJ,12852,171.0,Rio de Janeiro,75.158,0.013
0,AC,81,1.0,Acre,81.000,0.012
7,ES,2033,23.0,Espírito Santo,88.391,0.011
